In [ ]:
# ============================================================================
# ConvLSTM Model - Riverbank Prediction (QUARTERLY ANALYSIS)
# Multi-Sequence Search (6, 8, 10) → 200 Epochs + Early Stopping
# STRICT Temporal Split (Triple-Layer Leak Prevention)
# ============================================================================

# ── Suppress TF / XLA / CUDA warnings BEFORE importing tensorflow ──
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['XLA_FLAGS'] = '--xla_gpu_strict_conv_algorithm_picker=false'
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'

import re
import glob
import sys
import logging
import numpy as np
import pandas as pd

logging.getLogger('tensorflow').setLevel(logging.FATAL)
logging.getLogger('absl').setLevel(logging.FATAL)

import tensorflow as tf
tf.get_logger().setLevel('FATAL')
tf.autograph.set_verbosity(0)

import warnings
warnings.filterwarnings('ignore')
import absl.logging
absl.logging.set_verbosity(absl.logging.FATAL)
absl.logging._warn_preinit_stderr = False

# ── FIX: Enable unsafe deserialization to avoid Lambda layer error ──
try:
    import keras
    keras.config.enable_unsafe_deserialization()
except AttributeError:
    try:
        tf.keras.config.enable_unsafe_deserialization()
    except AttributeError:
        pass

from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback
)
from tensorflow.keras.optimizers import Adam
import rasterio
from sklearn.metrics import precision_score, recall_score
from skimage.transform import resize
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

np.random.seed(42)
tf.random.set_seed(42)

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU configured: {len(gpus)} GPU(s) available")
    except RuntimeError as e:
        print(f"GPU Error: {e}")
else:
    print("⚠ No GPU found, using CPU")

# GPU warmup
_warmup = tf.zeros((1, 1), dtype=tf.float32)
_ = tf.matmul(_warmup, _warmup)
del _warmup

# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_DIR = "/kaggle/input/datasets/kaoserahamed/quarterlygapfilled"
IMG_HEIGHT = 256
IMG_WIDTH = 256
SEQUENCE_LENGTHS = [6, 8, 10]           # ← Updated for more images
PREDICTION_HORIZON = 1
STRIDE = 1
EPOCHS = 200
EARLY_STOP_PATIENCE = 20
REDUCE_LR_PATIENCE = 7
BATCH_SIZE = 4
N_COMPONENTS = 3

# Quarterly cutoffs (float): year + (quarter-1)*0.25
# 2015.0 = end of 2014-Q4  →  test starts from 2015-Q1
# 2020.0 = end of 2019-Q4  →  test starts from 2020-Q1
SETUP_CONFIGS = {
    'Setup 1': {'cutoff_yq': 2015.0, 'test_label': 'Test: 2015-Q1 → latest'},
    'Setup 2': {'cutoff_yq': 2020.0, 'test_label': 'Test: 2020-Q1 → latest'},
}

# ============================================================================
# QUARTERLY PARSING UTILITIES
# ============================================================================

MONTH_TO_QUARTER = {
    1: 1, 2: 1, 3: 1,
    4: 2, 5: 2, 6: 2,
    7: 3, 8: 3, 9: 3,
    10: 4, 11: 4, 12: 4,
}


def parse_year_quarter(filename):
    """
    Extract year and quarter from filename.
    Supports: YYYY_QN, YYYYQN, YYYY-MM-DD, YYYY_MM, YYYY (defaults Q1)
    """
    name = os.path.basename(filename)

    # Pattern 1: explicit quarter  e.g., 2015_Q2, 2015Q3
    m = re.search(r'(\d{4})[_\-]?[Qq](\d)', name)
    if m:
        return int(m.group(1)), int(m.group(2))

    # Pattern 2: YYYY-MM-DD or YYYY_MM_DD
    m = re.search(r'(\d{4})[_\-](\d{2})[_\-](\d{2})', name)
    if m:
        year, month = int(m.group(1)), int(m.group(2))
        return year, MONTH_TO_QUARTER.get(month, 1)

    # Pattern 3: YYYY_MM (no day)
    m = re.search(r'(\d{4})[_\-](\d{2})', name)
    if m:
        year, month = int(m.group(1)), int(m.group(2))
        if 1 <= month <= 12:
            return year, MONTH_TO_QUARTER.get(month, 1)

    # Pattern 4: year only → default Q1
    m = re.search(r'(\d{4})', name)
    if m:
        return int(m.group(1)), 1

    return None


def year_quarter_to_float(year, quarter):
    """2020-Q3 → 2020.50"""
    return year + (quarter - 1) * 0.25


def float_to_year_quarter(yq_float):
    """2020.50 → (2020, 3)"""
    year = int(yq_float)
    quarter = int(round((yq_float - year) / 0.25)) + 1
    quarter = max(1, min(4, quarter))
    return year, quarter


def yq_label(yq_float):
    """2020.5 → '2020-Q3'"""
    y, q = float_to_year_quarter(yq_float)
    return f"{y}-Q{q}"


# ============================================================================
# PREPROCESSING FUNCTIONS
# ============================================================================

def keep_largest_n_components_cv2(mask, n=1):
    binary_mask = (mask > 0).astype(np.uint8)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary_mask, connectivity=8
    )
    if num_labels <= 1:
        return np.zeros_like(mask)
    areas = stats[1:, cv2.CC_STAT_AREA]
    sorted_indices = np.argsort(areas)[::-1]
    n_to_keep = min(n, len(sorted_indices))
    top_n_indices = sorted_indices[:n_to_keep]
    cleaned_mask = np.zeros_like(mask)
    for idx in top_n_indices:
        actual_label = idx + 1
        cleaned_mask[labels == actual_label] = mask.max()
    return cleaned_mask


def load_and_preprocess_image(filepath, apply_cleaning=True,
                              n_components=N_COMPONENTS):
    with rasterio.open(filepath) as src:
        img = src.read(1)
    if apply_cleaning:
        img = keep_largest_n_components_cv2(img, n=n_components)
    img_resized = resize(
        img, (IMG_HEIGHT, IMG_WIDTH),
        mode='constant', preserve_range=True, anti_aliasing=False
    )
    img_binary = (img_resized > 0.5).astype(np.float32)
    return img_binary


def create_sequences_with_yq(images, yq_floats, seq_len):
    """
    Create sequences. Also stores the raw image indices used in each
    sequence+target so we can verify no image appears in both
    train and test.
    """
    X, y = [], []
    input_yq_list, target_yq_list = [], []
    input_idx_list, target_idx_list = [], []

    for i in range(0, len(images) - seq_len - PREDICTION_HORIZON + 1, STRIDE):
        X.append(images[i:i + seq_len])
        y.append(images[i + seq_len])
        input_yq_list.append(yq_floats[i:i + seq_len])
        target_yq_list.append(yq_floats[i + seq_len])
        input_idx_list.append(list(range(i, i + seq_len)))
        target_idx_list.append(i + seq_len)

    return (np.array(X), np.array(y),
            np.array(input_yq_list), np.array(target_yq_list),
            input_idx_list, np.array(target_idx_list))


# ============================================================================
# LOSSES AND METRICS
# ============================================================================

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (
        K.sum(y_true_f) + K.sum(y_pred_f) + smooth
    )


def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)


def combined_loss(y_true, y_pred):
    bce = K.mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)


def iou_metric(y_true, y_pred, threshold=0.5):
    y_pred_binary = K.cast(y_pred > threshold, dtype='float32')
    y_true_binary = K.cast(y_true > threshold, dtype='float32')
    intersection = K.sum(y_true_binary * y_pred_binary)
    union = K.sum(y_true_binary) + K.sum(y_pred_binary) - intersection
    return (intersection + K.epsilon()) / (union + K.epsilon())


def dice_coefficient_np(y_true, y_pred, threshold=0.5):
    y_true_b = (y_true > threshold).astype(np.float32)
    y_pred_b = (y_pred > threshold).astype(np.float32)
    intersection = np.sum(y_true_b * y_pred_b)
    return (2. * intersection) / (np.sum(y_true_b) + np.sum(y_pred_b) + 1e-6)


def iou_np(y_true, y_pred, threshold=0.5):
    y_true_b = (y_true > threshold).astype(np.float32)
    y_pred_b = (y_pred > threshold).astype(np.float32)
    intersection = np.sum(y_true_b * y_pred_b)
    union = np.sum(y_true_b) + np.sum(y_pred_b) - intersection
    return intersection / (union + 1e-6)


def calculate_area_difference(y_true, y_pred, pixel_area_km2, threshold=0.5):
    true_area = np.sum((y_true > threshold).astype(np.float32)) * pixel_area_km2
    pred_area = np.sum((y_pred > threshold).astype(np.float32)) * pixel_area_km2
    return pred_area - true_area


# ============================================================================
# CUSTOM STRUCTURED TRAINING CALLBACK
# ============================================================================

class StructuredTrainingLogger(Callback):
    def __init__(self, total_epochs):
        super().__init__()
        self.total_epochs = total_epochs
        self.best_val_loss = np.inf

    def on_train_begin(self, logs=None):
        header = (
            f"{'Ep':>4s}/{'Tot':<4s} │ {'Loss':>8s} │ {'VLoss':>8s} │ "
            f"{'Dice':>6s} │ {'VDice':>6s} │ {'IoU':>6s} │ "
            f"{'VIoU':>6s} │ {'LR':>9s} │ Note"
        )
        print("─" * len(header))
        print(header)
        print("─" * len(header))

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = logs.get('loss', 0)
        vloss = logs.get('val_loss', 0)
        dice = logs.get('dice_coefficient', 0)
        vdice = logs.get('val_dice_coefficient', 0)
        iou = logs.get('iou_metric', 0)
        viou = logs.get('val_iou_metric', 0)
        lr = float(K.get_value(self.model.optimizer.learning_rate))

        note = ""
        if vloss < self.best_val_loss:
            self.best_val_loss = vloss
            note = "★ saved"

        print(
            f"{epoch+1:>4d}/{self.total_epochs:<4d} │ {loss:>8.4f} │ "
            f"{vloss:>8.4f} │ {dice:>6.4f} │ {vdice:>6.4f} │ "
            f"{iou:>6.4f} │ {viou:>6.4f} │ {lr:>9.2e} │ {note}"
        )

    def on_train_end(self, logs=None):
        print("─" * 85)
        print(f"  Training finished  │  Best val_loss: {self.best_val_loss:.4f}")
        print("─" * 85)


# ============================================================================
# CALLBACKS FACTORY
# ============================================================================

def create_callbacks(model_name, checkpoint_dir='checkpoints'):
    os.makedirs(checkpoint_dir, exist_ok=True)
    ckpt_path = os.path.join(checkpoint_dir, f'{model_name}_best.keras')
    return [
        ModelCheckpoint(
            ckpt_path, monitor='val_loss', save_best_only=True,
            save_weights_only=False, mode='min', verbose=0
        ),
        EarlyStopping(
            monitor='val_loss', patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True, verbose=0
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=REDUCE_LR_PATIENCE, min_lr=1e-7, verbose=0
        ),
        StructuredTrainingLogger(total_epochs=EPOCHS),
    ]


# ============================================================================
# MODEL DEFINITION (no Lambda layers)
# ============================================================================

def build_convlstm_model(seq_len):
    input_shape = (seq_len, IMG_HEIGHT, IMG_WIDTH, 1)
    inputs = layers.Input(shape=input_shape)

    x = layers.TimeDistributed(
        layers.Conv2D(32, 3, padding='same', activation='relu'))(inputs)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.MaxPooling2D(2))(x)
    x = layers.TimeDistributed(layers.Dropout(0.2))(x)

    x = layers.TimeDistributed(
        layers.Conv2D(64, 3, padding='same', activation='relu'))(x)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.MaxPooling2D(2))(x)
    x = layers.TimeDistributed(layers.Dropout(0.2))(x)

    x = layers.ConvLSTM2D(
        filters=128, kernel_size=(3, 3), padding='same',
        return_sequences=True, dropout=0.3,
        kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)

    x = layers.ConvLSTM2D(
        filters=64, kernel_size=(3, 3), padding='same',
        return_sequences=False, dropout=0.3,
        kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)

    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(16, 3, padding='same', activation='relu')(x)
    outputs = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name=f'ConvLSTM_seq{seq_len}')
    model.compile(
        optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
        loss=combined_loss,
        metrics=[dice_coefficient, iou_metric]
    )
    return model


# ============================================================================
# STRICT TEMPORAL SPLIT – TRIPLE-LAYER LEAK PREVENTION
# ============================================================================

def verify_no_data_leak(train_input_indices, train_target_indices,
                        val_input_indices, val_target_indices,
                        test_input_indices, test_target_indices,
                        train_yq, val_yq, test_yq,
                        seq_len, cutoff_yq):
    """
    Three layers of leak verification:
      1) Quarter-level:  no target quarter appears in both train/val and test
      2) Image-index-level: no raw image index is used by both
         train/val sequences AND test sequences
      3) Gap check: the earliest test input quarter must be AFTER
         the latest train/val target quarter
    """
    errors = []

    # ── Layer 1: Target quarter overlap ──
    trainval_target_yq = set(train_yq.tolist()) | set(val_yq.tolist())
    test_target_yq = set(test_yq.tolist())
    overlap_yq = trainval_target_yq & test_target_yq
    if overlap_yq:
        labels = sorted([yq_label(q) for q in overlap_yq])
        errors.append(
            f"LAYER-1 LEAK: Target quarter overlap → {labels}"
        )

    # ── Layer 2: Raw image index overlap ──
    trainval_all_idx = set()
    for idx_list in train_input_indices + val_input_indices:
        trainval_all_idx.update(idx_list)
    trainval_all_idx.update(train_target_indices.tolist())
    trainval_all_idx.update(val_target_indices.tolist())

    test_all_idx = set()
    for idx_list in test_input_indices:
        test_all_idx.update(idx_list)
    test_all_idx.update(test_target_indices.tolist())

    overlap_idx = trainval_all_idx & test_all_idx
    if overlap_idx:
        errors.append(
            f"LAYER-2 LEAK: {len(overlap_idx)} shared image indices "
            f"between train/val and test → {sorted(overlap_idx)[:10]}..."
        )

    # ── Layer 3: Temporal gap check ──
    max_trainval_target = max(trainval_target_yq) if trainval_target_yq else -1
    # Find the earliest quarter used in ANY test sequence (input or target)
    min_test_input_yq = float('inf')
    for idx_list in test_input_indices:
        # These are raw image indices; we'd need the yq_floats for them.
        # Instead, we verify via the test_target_yq minus seq_len window.
        pass

    # Simpler gap check: max train/val target < min test target - 0
    min_test_target = min(test_target_yq) if test_target_yq else float('inf')
    if max_trainval_target >= min_test_target:
        errors.append(
            f"LAYER-3 LEAK: Latest train/val target "
            f"({yq_label(max_trainval_target)}) >= earliest test target "
            f"({yq_label(min_test_target)})"
        )

    if errors:
        for e in errors:
            print(f"  ✗ {e}")
        raise ValueError(
            f"DATA LEAK DETECTED ({len(errors)} violations)! "
            f"See details above."
        )

    print(f"  ✓ Layer-1: No target quarter overlap")
    print(f"  ✓ Layer-2: No shared image indices "
          f"(train/val uses {len(trainval_all_idx)} imgs, "
          f"test uses {len(test_all_idx)} imgs)")
    print(f"  ✓ Layer-3: Temporal gap OK "
          f"(last train/val target={yq_label(max_trainval_target)}, "
          f"first test target={yq_label(min_test_target)})")


def prepare_split_quarterly(X_all, y_all, target_yq_all, input_yq_all,
                            input_idx_all, target_idx_all,
                            cutoff_yq, seq_len, all_yq_floats):
    """
    Strict temporal split with triple-layer leak prevention.

    Key guarantee: Every image index used by a test sequence (both in its
    input window AND its target) must have a quarter strictly AFTER the
    cutoff.  To achieve this for long sequences, we enforce:
        - For TRAIN: max input quarter ≤ cutoff AND target quarter ≤ cutoff
        - For TEST:  target quarter > cutoff
        - ADDITIONAL: no test sequence's input window may include any
          quarter ≤ cutoff (strict gap enforcement)
    """
    max_input_yq = np.array([iyq.max() for iyq in input_yq_all])
    min_input_yq = np.array([iyq.min() for iyq in input_yq_all])

    # ── Train mask: all input quarters AND target quarter ≤ cutoff ──
    train_mask = (
        (target_yq_all <= cutoff_yq) &
        (max_input_yq <= cutoff_yq)
    )

    # ── Test mask: target > cutoff AND all input quarters > cutoff ──
    # This is the STRICT version: even the oldest image in the input
    # window must be after the cutoff, guaranteeing zero image sharing.
    test_mask_strict = (
        (target_yq_all > cutoff_yq) &
        (min_input_yq > cutoff_yq)
    )

    # If strict is too restrictive (e.g., few images after cutoff),
    # fall back to standard: target > cutoff only, but still verify.
    test_mask_standard = (target_yq_all > cutoff_yq)

    # Use strict if it yields enough samples, otherwise standard
    if test_mask_strict.sum() >= 1:
        test_mask = test_mask_strict
        gap_mode = "STRICT (all input quarters > cutoff)"
    else:
        test_mask = test_mask_standard
        gap_mode = "STANDARD (target > cutoff, input may overlap)"
        print(f"  ⚠ Strict gap mode yielded 0 test samples; "
              f"using standard mode")

    X_train = X_all[train_mask]
    y_train = y_all[train_mask]
    ty_train = target_yq_all[train_mask]
    train_input_idx = [input_idx_all[i] for i in range(len(train_mask))
                       if train_mask[i]]
    train_target_idx = target_idx_all[train_mask]

    X_test = X_all[test_mask]
    y_test = y_all[test_mask]
    ty_test = target_yq_all[test_mask]
    test_input_idx = [input_idx_all[i] for i in range(len(test_mask))
                      if test_mask[i]]
    test_target_idx = target_idx_all[test_mask]

    if len(X_train) == 0 or len(X_test) == 0:
        raise ValueError(
            f"Empty split! Train={len(X_train)}, Test={len(X_test)} "
            f"with cutoff={yq_label(cutoff_yq)}, seq_len={seq_len}"
        )

    # ── Sort training chronologically ──
    sort_idx = np.argsort(ty_train)
    X_train = X_train[sort_idx]
    y_train = y_train[sort_idx]
    ty_train = ty_train[sort_idx]
    train_input_idx = [train_input_idx[i] for i in sort_idx]
    train_target_idx = train_target_idx[sort_idx]

    # ── 85/15 train/val split (chronological) ──
    split_idx = int(len(X_train) * 0.85)
    split_idx = max(1, min(split_idx, len(X_train) - 1))

    X_tr  = X_train[:split_idx]
    y_tr  = y_train[:split_idx]
    ty_tr = ty_train[:split_idx]
    tr_input_idx  = train_input_idx[:split_idx]
    tr_target_idx = train_target_idx[:split_idx]

    X_val  = X_train[split_idx:]
    y_val  = y_train[split_idx:]
    ty_val = ty_train[split_idx:]
    val_input_idx  = train_input_idx[split_idx:]
    val_target_idx = train_target_idx[split_idx:]

    if len(X_val) == 0:
        raise ValueError("Empty validation set!")

    # ── Triple-layer leak verification ──
    print(f"\n  Gap mode: {gap_mode}")
    verify_no_data_leak(
        tr_input_idx, tr_target_idx,
        val_input_idx, val_target_idx,
        test_input_idx, test_target_idx,
        ty_tr, ty_val, ty_test,
        seq_len, cutoff_yq
    )

    # ── Add channel dimension ──
    X_tr   = np.expand_dims(X_tr,   axis=-1)
    y_tr   = np.expand_dims(y_tr,   axis=-1)
    X_val  = np.expand_dims(X_val,  axis=-1)
    y_val  = np.expand_dims(y_val,  axis=-1)
    X_test = np.expand_dims(X_test, axis=-1)
    y_test = np.expand_dims(y_test, axis=-1)

    print(f"\n  Train : {len(X_tr):>4d} samples  "
          f"({yq_label(ty_tr.min())} → {yq_label(ty_tr.max())})")
    print(f"  Val   : {len(X_val):>4d} samples  "
          f"({yq_label(ty_val.min())} → {yq_label(ty_val.max())})")
    print(f"  Test  : {len(X_test):>4d} samples  "
          f"({yq_label(ty_test.min())} → {yq_label(ty_test.max())})")
    print(f"  Total images in dataset: {len(all_yq_floats)}")

    return X_tr, y_tr, X_val, y_val, X_test, y_test, ty_test


# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_model(model, X_test, y_test, target_yq_test,
                   pixel_area_km2, model_name, setup_name):
    pred = model.predict(X_test, verbose=0)
    persistence_pred = X_test[:, -1, :, :, :]

    results = []
    for name, predictions in [('Persistence', persistence_pred),
                               (model_name, pred)]:
        for i in range(len(y_test)):
            yt = y_test[i, :, :, 0]
            yp = predictions[i, :, :, 0]
            yq = target_yq_test[i]
            yq_str = yq_label(yq)

            iou = iou_np(yt, yp)
            dice = dice_coefficient_np(yt, yp)
            yt_flat = (yt.flatten() > 0.5).astype(int)
            yp_flat = (yp.flatten() > 0.5).astype(int)
            prec = precision_score(yt_flat, yp_flat, zero_division=0)
            rec  = recall_score(yt_flat, yp_flat, zero_division=0)
            area_diff = calculate_area_difference(yt, yp, pixel_area_km2)

            results.append({
                'Setup': setup_name, 'Model': name,
                'YQ_float': yq, 'Quarter': yq_str,
                'IoU': iou, 'Dice': dice, 'Precision': prec,
                'Recall': rec, 'Area_Diff_km2': area_diff
            })

    return pd.DataFrame(results)


# ============================================================================
# DATA LOADING
# ============================================================================

print("=" * 80)
print("LOADING DATA (QUARTERLY ANALYSIS)")
print("=" * 80)

files = sorted(glob.glob(os.path.join(DATA_DIR, "*.tif")))
file_info = []
for filepath in files:
    filename = os.path.basename(filepath)
    parsed = parse_year_quarter(filename)
    if parsed:
        year, quarter = parsed
        file_info.append({
            'filepath': filepath,
            'filename': filename,
            'year': year,
            'quarter': quarter,
            'yq_float': year_quarter_to_float(year, quarter),
            'yq_label': f"{year}-Q{quarter}",
        })

df_files = pd.DataFrame(file_info).sort_values('yq_float').reset_index(drop=True)
print(f"Found {len(df_files)} files")
print(f"  Year range   : {df_files['year'].min()} → {df_files['year'].max()}")
print(f"  Quarter range: {df_files['yq_label'].iloc[0]} → "
      f"{df_files['yq_label'].iloc[-1]}")

q_counts = df_files.groupby('quarter').size()
print(f"  Quarter distribution: {dict(q_counts)}")

unique_quarters = df_files['quarter'].unique()
if len(unique_quarters) == 1 and unique_quarters[0] == 1:
    print("\n  ⚠  WARNING: All files parsed as Q1 — filenames likely "
          "contain only YYYY.")
    print("     Quarterly framework still runs, but each 'quarter' = "
          "one year.")
    print("     For true quarterly analysis, include month or quarter "
          "in filenames.")
    print("     Supported patterns: YYYY_QN, YYYY-MM-DD, YYYY_MM\n")

# ── Minimum data check for largest sequence length ──
max_seq = max(SEQUENCE_LENGTHS)
min_required = max_seq + PREDICTION_HORIZON
if len(df_files) < min_required:
    print(f"\n  ✗ FATAL: Need at least {min_required} images for "
          f"seq_len={max_seq}, but only {len(df_files)} found.")
    sys.exit(1)
else:
    print(f"  ✓ Sufficient data: {len(df_files)} images ≥ "
          f"{min_required} required for seq_len={max_seq}")

all_images, all_yq_floats = [], []
for idx, row in df_files.iterrows():
    img = load_and_preprocess_image(row['filepath'])
    all_images.append(img)
    all_yq_floats.append(row['yq_float'])
    if (idx + 1) % 10 == 0:
        print(f"  Processed {idx+1}/{len(df_files)} images...")

print(f"✓ Loaded {len(all_images)} images")

# Pixel area calculation
with rasterio.open(df_files.iloc[0]['filepath']) as src:
    bounds = src.bounds
    center_lat = (bounds.top + bounds.bottom) / 2
    if src.crs and src.crs.to_epsg() == 4326:
        meters_per_deg_lon = 111320 * np.cos(np.radians(center_lat))
        width_m  = (bounds.right - bounds.left) * meters_per_deg_lon
        height_m = (bounds.top - bounds.bottom) * 111320
        total_area_km2 = (width_m * height_m) / 1e6
    else:
        total_area_km2 = (
            (bounds.right - bounds.left) *
            (bounds.top - bounds.bottom)
        ) / 1e6

pixel_area_km2 = total_area_km2 / (IMG_HEIGHT * IMG_WIDTH)
print(f"✓ Pixel area: {pixel_area_km2:.8f} km²")

# ============================================================================
# MULTI-SEQUENCE SEARCH + BOTH SETUPS
# ============================================================================

all_results = []
summary_rows = []
best_configs = {}
training_histories = {}

for seq_len in SEQUENCE_LENGTHS:
    print("\n" + "█" * 80)
    print(f"  SEQUENCE LENGTH = {seq_len}")
    print("█" * 80)

    (X_all_seq, y_all_seq, input_yq_all, target_yq_all,
     input_idx_all, target_idx_all) = \
        create_sequences_with_yq(all_images, all_yq_floats, seq_len)

    print(f"✓ {len(X_all_seq)} sequences  |  target quarters "
          f"{yq_label(target_yq_all.min())} → "
          f"{yq_label(target_yq_all.max())}")

    for setup_name, cfg in SETUP_CONFIGS.items():
        cutoff_yq = cfg['cutoff_yq']
        label = cfg['test_label']
        tag = f"seq{seq_len}_{setup_name.replace(' ', '')}"

        print(f"\n{'─'*70}")
        print(f"  {setup_name} (cutoff={yq_label(cutoff_yq)})  |  "
              f"SeqLen={seq_len}")
        print(f"{'─'*70}")

        try:
            X_tr, y_tr, X_val, y_val, X_test, y_test, ty_test = \
                prepare_split_quarterly(
                    X_all_seq, y_all_seq, target_yq_all,
                    input_yq_all, input_idx_all, target_idx_all,
                    cutoff_yq, seq_len, all_yq_floats
                )
        except ValueError as e:
            print(f"  ⚠ Skipping: {e}")
            continue

        # ── Build & train ──
        K.clear_session()
        tf.random.set_seed(42)
        np.random.seed(42)

        model = build_convlstm_model(seq_len)
        if (seq_len == SEQUENCE_LENGTHS[0] and
                setup_name == list(SETUP_CONFIGS.keys())[0]):
            model.summary()

        print(f"\n  Training {tag}  (max {EPOCHS} epochs, "
              f"early-stop patience={EARLY_STOP_PATIENCE})\n")

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=create_callbacks(tag),
            verbose=0
        )
        training_histories[tag] = history

        stopped_epoch = len(history.history['loss'])
        best_val_loss = min(history.history['val_loss'])
        best_val_dice = max(history.history['val_dice_coefficient'])
        best_val_iou  = max(history.history['val_iou_metric'])

        print(f"\n  ✓ Stopped at epoch {stopped_epoch}/{EPOCHS}  |  "
              f"Best val_loss={best_val_loss:.4f}  "
              f"val_dice={best_val_dice:.4f}  "
              f"val_iou={best_val_iou:.4f}")

        # ── Evaluate ──
        model_label = f'ConvLSTM(seq={seq_len})'
        df_res = evaluate_model(
            model, X_test, y_test, ty_test,
            pixel_area_km2, model_label,
            f'{setup_name} ({label})'
        )
        df_res['SeqLen'] = seq_len
        all_results.append(df_res)

        for mdl_name in df_res['Model'].unique():
            sub = df_res[df_res['Model'] == mdl_name]
            summary_rows.append({
                'SeqLen': seq_len,
                'Setup': setup_name,
                'Model': mdl_name,
                'Epochs_Run': stopped_epoch,
                'Best_Val_Loss': (
                    best_val_loss if mdl_name != 'Persistence'
                    else np.nan
                ),
                'IoU': sub['IoU'].mean(),
                'Dice': sub['Dice'].mean(),
                'Precision': sub['Precision'].mean(),
                'Recall': sub['Recall'].mean(),
                'Area_Diff_km2': sub['Area_Diff_km2'].mean(),
                'Abs_Area_Diff_km2': sub['Area_Diff_km2'].abs().mean(),
            })


# ============================================================================
# AGGREGATE RESULTS
# ============================================================================

df_all = pd.concat(all_results, ignore_index=True)
df_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 100)
print("  SEQUENCE LENGTH COMPARISON  –  Mean Test Metrics (ConvLSTM only)")
print("=" * 100)

convlstm_summary = df_summary[df_summary['Model'] != 'Persistence'].copy()
pivot = convlstm_summary.pivot_table(
    index='SeqLen', columns='Setup',
    values=['IoU', 'Dice', 'Precision', 'Recall', 'Abs_Area_Diff_km2'],
    aggfunc='mean'
).round(4)
print(pivot.to_string())

# ============================================================================
# IDENTIFY BEST SEQUENCE LENGTH PER SETUP
# ============================================================================

print("\n" + "=" * 100)
print("  BEST SEQUENCE LENGTH PER SETUP  (by mean IoU on test)")
print("=" * 100)

for setup_name in SETUP_CONFIGS:
    sub = convlstm_summary[convlstm_summary['Setup'] == setup_name]
    if sub.empty:
        continue
    best_row = sub.loc[sub['IoU'].idxmax()]
    best_configs[setup_name] = int(best_row['SeqLen'])
    print(f"  {setup_name}: Best SeqLen = {int(best_row['SeqLen'])}  "
          f"(IoU={best_row['IoU']:.4f}, Dice={best_row['Dice']:.4f})")

# ============================================================================
# SIDE-BY-SIDE: ACTUAL vs PREDICTED
# ============================================================================

print("\n" + "=" * 100)
print("  SIDE-BY-SIDE  –  Actual vs Predicted  (Best Config per Setup)")
print("=" * 100)

for setup_name, cfg in SETUP_CONFIGS.items():
    best_seq = best_configs.get(setup_name)
    if best_seq is None:
        continue

    label = cfg['test_label']
    sub = df_all[
        (df_all['Setup'] == f'{setup_name} ({label})') &
        (df_all['SeqLen'] == best_seq)
    ].copy()

    if sub.empty:
        continue

    print(f"\n{'━'*90}")
    print(f"  {setup_name} | Best Sequence Length = {best_seq} | {label}")
    print(f"{'━'*90}")

    quarters = sorted(sub['YQ_float'].unique())
    models_in_sub = sub['Model'].unique()

    rows = []
    for yq in quarters:
        row = {'Quarter': yq_label(yq)}
        for mdl in models_in_sub:
            m = sub[(sub['YQ_float'] == yq) & (sub['Model'] == mdl)]
            if m.empty:
                continue
            m = m.iloc[0]
            prefix = 'Pred' if 'ConvLSTM' in mdl else 'Pers'
            row[f'{prefix}_IoU']       = round(m['IoU'], 4)
            row[f'{prefix}_Dice']      = round(m['Dice'], 4)
            row[f'{prefix}_Precision'] = round(m['Precision'], 4)
            row[f'{prefix}_Recall']    = round(m['Recall'], 4)
            row[f'{prefix}_ΔArea_km²'] = round(m['Area_Diff_km2'], 6)
        rows.append(row)

    df_side = pd.DataFrame(rows)

    desired_order = ['Quarter']
    for prefix in ['Pred', 'Pers']:
        for metric in ['IoU', 'Dice', 'Precision', 'Recall', 'ΔArea_km²']:
            col = f'{prefix}_{metric}'
            if col in df_side.columns:
                desired_order.append(col)
    df_side = df_side[[c for c in desired_order if c in df_side.columns]]

    print(df_side.to_string(index=False))

    numeric_cols = [c for c in df_side.columns if c != 'Quarter']
    means = df_side[numeric_cols].mean()
    print(f"{'─'*90}")
    mean_str = "  MEAN |"
    for c in numeric_cols:
        mean_str += f"  {c}={means[c]:.4f}"
    print(mean_str)

# ============================================================================
# FULL SUMMARY TABLE
# ============================================================================

print("\n" + "=" * 100)
print("  FULL SUMMARY TABLE")
print("=" * 100)

display_cols = [
    'SeqLen', 'Setup', 'Model', 'Epochs_Run',
    'IoU', 'Dice', 'Precision', 'Recall',
    'Area_Diff_km2', 'Abs_Area_Diff_km2'
]
print(df_summary[display_cols].round(4).to_string(index=False))

# ============================================================================
# VISUALIZATION: TRAINING CURVES
# ============================================================================

fig, axes = plt.subplots(
    len(SETUP_CONFIGS), 3,
    figsize=(18, 5 * len(SETUP_CONFIGS))
)
if len(SETUP_CONFIGS) == 1:
    axes = axes[np.newaxis, :]

for row_idx, (setup_name, cfg) in enumerate(SETUP_CONFIGS.items()):
    best_seq = best_configs.get(setup_name)
    if best_seq is None:
        continue
    tag = f"seq{best_seq}_{setup_name.replace(' ', '')}"
    hist = training_histories.get(tag)
    if hist is None:
        continue

    ax = axes[row_idx, 0]
    ax.plot(hist.history['loss'], label='Train Loss')
    ax.plot(hist.history['val_loss'], label='Val Loss')
    ax.set_title(f'{setup_name} | seq={best_seq} | Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(True)

    ax = axes[row_idx, 1]
    ax.plot(hist.history['dice_coefficient'], label='Train Dice')
    ax.plot(hist.history['val_dice_coefficient'], label='Val Dice')
    ax.set_title(f'{setup_name} | seq={best_seq} | Dice')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Dice')
    ax.legend(); ax.grid(True)

    ax = axes[row_idx, 2]
    ax.plot(hist.history['iou_metric'], label='Train IoU')
    ax.plot(hist.history['val_iou_metric'], label='Val IoU')
    ax.set_title(f'{setup_name} | seq={best_seq} | IoU')
    ax.set_xlabel('Epoch'); ax.set_ylabel('IoU')
    ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig('training_curves_best_configs_quarterly.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✓ Training curves saved → training_curves_best_configs_quarterly.png")

# ============================================================================
# VISUALIZATION: BAR CHART – Sequence Length Comparison
# ============================================================================

fig, axes = plt.subplots(
    1, len(SETUP_CONFIGS),
    figsize=(8 * len(SETUP_CONFIGS), 6)
)
if len(SETUP_CONFIGS) == 1:
    axes = [axes]

metrics_to_plot = ['IoU', 'Dice', 'Precision', 'Recall']

for ax_idx, (setup_name, cfg) in enumerate(SETUP_CONFIGS.items()):
    ax = axes[ax_idx]
    sub = convlstm_summary[convlstm_summary['Setup'] == setup_name]
    if sub.empty:
        continue

    x = np.arange(len(metrics_to_plot))
    width = 0.2
    for i, sl in enumerate(SEQUENCE_LENGTHS):
        vals = sub[sub['SeqLen'] == sl][metrics_to_plot].values
        if len(vals) == 0:
            continue
        vals = vals[0]
        offset = (i - len(SEQUENCE_LENGTHS) / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=f'seq={sl}')
        for bar, v in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8
            )

    best = best_configs.get(setup_name)
    ax.set_title(f'{setup_name} – Sequence Comparison (best={best})')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_to_plot)
    ax.set_ylim(0, 1.1)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('sequence_comparison_bar_quarterly.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✓ Sequence comparison chart saved → "
      "sequence_comparison_bar_quarterly.png")

# ============================================================================
# VISUALIZATION: SAMPLE PREDICTIONS
# ============================================================================

CUSTOM_OBJECTS = {
    'combined_loss': combined_loss,
    'dice_coefficient': dice_coefficient,
    'iou_metric': iou_metric,
}

for setup_name, cfg in SETUP_CONFIGS.items():
    best_seq = best_configs.get(setup_name)
    if best_seq is None:
        continue

    cutoff_yq = cfg['cutoff_yq']
    label = cfg['test_label']
    tag = f"seq{best_seq}_{setup_name.replace(' ', '')}"

    (X_all_v, y_all_v, iy_all_v, ty_all_v,
     _, _) = create_sequences_with_yq(
        all_images, all_yq_floats, best_seq
    )

    # Use same strict test mask
    min_input_yq_v = np.array([iyq.min() for iyq in iy_all_v])
    test_mask_strict = (ty_all_v > cutoff_yq) & (min_input_yq_v > cutoff_yq)
    test_mask_std = (ty_all_v > cutoff_yq)
    test_mask = test_mask_strict if test_mask_strict.sum() >= 1 else test_mask_std

    X_test_v  = np.expand_dims(X_all_v[test_mask], axis=-1)
    y_test_v  = y_all_v[test_mask]
    ty_test_v = ty_all_v[test_mask]

    ckpt = f'checkpoints/{tag}_best.keras'
    if os.path.exists(ckpt):
        try:
            best_model = tf.keras.models.load_model(
                ckpt, custom_objects=CUSTOM_OBJECTS,
                safe_mode=False,
            )
        except TypeError:
            best_model = tf.keras.models.load_model(
                ckpt, custom_objects=CUSTOM_OBJECTS,
            )
    else:
        print(f"  ⚠ Checkpoint not found: {ckpt}, skipping visualisation")
        continue

    preds = best_model.predict(X_test_v, verbose=0)
    n_show = min(6, len(y_test_v))
    indices = np.linspace(0, len(y_test_v) - 1, n_show, dtype=int)

    fig, axes_v = plt.subplots(n_show, 3, figsize=(15, 4 * n_show))
    if n_show == 1:
        axes_v = axes_v[np.newaxis, :]

    for row, idx in enumerate(indices):
        actual    = y_test_v[idx]
        predicted = (preds[idx, :, :, 0] > 0.5).astype(np.float32)

        diff = np.zeros((*actual.shape, 3))
        diff[:, :, 1] = actual * (1 - predicted)      # Green = FN
        diff[:, :, 0] = predicted * (1 - actual)      # Red   = FP
        diff[:, :, 2] = actual * predicted             # Blue  = TP

        qtr_lbl  = yq_label(ty_test_v[idx])
        iou_val  = iou_np(actual, predicted)
        dice_val = dice_coefficient_np(actual, predicted)

        axes_v[row, 0].imshow(actual, cmap='gray')
        axes_v[row, 0].set_title(f'Actual – {qtr_lbl}', fontsize=11)
        axes_v[row, 0].axis('off')

        axes_v[row, 1].imshow(predicted, cmap='gray')
        axes_v[row, 1].set_title(
            f'Predicted – {qtr_lbl}\n'
            f'IoU={iou_val:.4f}  Dice={dice_val:.4f}',
            fontsize=11
        )
        axes_v[row, 1].axis('off')

        axes_v[row, 2].imshow(diff)
        axes_v[row, 2].set_title(
            f'Difference – {qtr_lbl}\n(R=FP, G=FN, B=TP)',
            fontsize=11
        )
        axes_v[row, 2].axis('off')

    fig.suptitle(
        f'{setup_name} | seq={best_seq} | '
        f'Actual vs Predicted (Quarterly)',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    fname = (f'predictions_{setup_name.replace(" ", "_")}'
             f'_seq{best_seq}_quarterly.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Prediction samples saved → {fname}")

# ============================================================================
# VISUALIZATION: QUARTERLY TREND PLOTS
# ============================================================================

print("\n" + "=" * 80)
print("  QUARTERLY TREND PLOTS")
print("=" * 80)

for setup_name, cfg in SETUP_CONFIGS.items():
    best_seq = best_configs.get(setup_name)
    if best_seq is None:
        continue

    label = cfg['test_label']
    sub = df_all[
        (df_all['Setup'] == f'{setup_name} ({label})') &
        (df_all['SeqLen'] == best_seq)
    ].copy()

    if sub.empty:
        continue

    convlstm_data = sub[
        sub['Model'].str.contains('ConvLSTM')
    ].sort_values('YQ_float')
    persist_data = sub[
        sub['Model'] == 'Persistence'
    ].sort_values('YQ_float')

    fig, axes_t = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(
        f'{setup_name} | seq={best_seq} | Quarterly Test Metrics',
        fontsize=14, fontweight='bold'
    )

    for ax, metric in zip(
        axes_t.flat, ['IoU', 'Dice', 'Precision', 'Recall']
    ):
        x_labels = [yq_label(yq) for yq in convlstm_data['YQ_float']]

        ax.plot(
            x_labels, convlstm_data[metric].values,
            'b-o', label=f'ConvLSTM(seq={best_seq})', markersize=5
        )
        if len(persist_data) > 0:
            ax.plot(
                x_labels, persist_data[metric].values,
                'r--s', label='Persistence', markersize=4, alpha=0.7
            )

        ax.set_title(metric, fontsize=12)
        ax.set_ylabel(metric)
        ax.set_xlabel('Quarter')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    fname = (f'quarterly_trends_{setup_name.replace(" ", "_")}'
             f'_seq{best_seq}.png')
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Quarterly trend plot saved → {fname}")

print("\n" + "=" * 100)
print("  ✓ ALL DONE – ConvLSTM multi-sequence QUARTERLY "
      "evaluation complete")
print("=" * 100)

E0000 00:00:1776574558.705431      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776574558.755854      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776574559.151340      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776574559.151393      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776574559.151396      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776574559.151398      55 computation_placer.cc:177] computation placer already registered. Please check linka

✓ GPU configured: 2 GPU(s) available
LOADING DATA (QUARTERLY ANALYSIS)
Found 152 files
  Year range   : 1988 → 2025
  Quarter range: 1988-Q1 → 2025-Q4
  Quarter distribution: {1: np.int64(38), 2: np.int64(38), 3: np.int64(38), 4: np.int64(38)}
  ✓ Sufficient data: 152 images ≥ 11 required for seq_len=10


I0000 00:00:1776574582.977209      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776574582.983371      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


  Processed 10/152 images...
  Processed 20/152 images...
  Processed 30/152 images...
  Processed 40/152 images...
  Processed 50/152 images...
  Processed 60/152 images...
  Processed 70/152 images...
  Processed 80/152 images...
  Processed 90/152 images...
  Processed 100/152 images...
  Processed 110/152 images...
  Processed 120/152 images...
  Processed 130/152 images...
  Processed 140/152 images...
  Processed 150/152 images...
✓ Loaded 152 images
✓ Pixel area: 0.37954857 km²

████████████████████████████████████████████████████████████████████████████████
  SEQUENCE LENGTH = 6
████████████████████████████████████████████████████████████████████████████████
✓ 146 sequences  |  target quarters 1989-Q3 → 2025-Q4

──────────────────────────────────────────────────────────────────────
  Setup 1 (cutoff=2015-Q1)  |  SeqLen=6
──────────────────────────────────────────────────────────────────────

  Gap mode: STRICT (all input quarters > cutoff)
  ✓ Layer-1: No target quarter overlap

Model: "ConvLSTM_seq6"

...

 Trainable params: 1,407,489 (5.37 MB)

 Non-trainable params: 768 (3.00 KB)


  Training seq6_Setup1  (max 200 epochs, early-stop patience=20)

──────────────────────────────────────────────────────────────────────────────────────
  Ep/Tot  │     Loss │    VLoss │   Dice │  VDice │    IoU │   VIoU │        LR │ Note
──────────────────────────────────────────────────────────────────────────────────────


I0000 00:00:1776574619.439223     127 service.cc:152] XLA service 0x7ca47000e200 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776574619.439290     127 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776574619.439301     127 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776574622.091340     127 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776574653.776500     127 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


   1/200  │   1.7136 │   1.6733 │ 0.1105 │ 0.0775 │ 0.1130 │ 0.0895 │  1.00e-04 │ ★ saved
   2/200  │   1.2840 │   1.3324 │ 0.2193 │ 0.1501 │ 0.3304 │ 0.3843 │  1.00e-04 │ ★ saved
   3/200  │   1.0308 │   1.0855 │ 0.3564 │ 0.2827 │ 0.4454 │ 0.4628 │  1.00e-04 │ ★ saved
   4/200  │   0.8856 │   0.9492 │ 0.4575 │ 0.3792 │ 0.4898 │ 0.4920 │  1.00e-04 │ ★ saved
   5/200  │   0.7886 │   0.8696 │ 0.5292 │ 0.4370 │ 0.5131 │ 0.4992 │  1.00e-04 │ ★ saved
   6/200  │   0.7211 │   0.8162 │ 0.5791 │ 0.4753 │ 0.5285 │ 0.5010 │  1.00e-04 │ ★ saved
   7/200  │   0.6741 │   0.7771 │ 0.6126 │ 0.5024 │ 0.5384 │ 0.4945 │  1.00e-04 │ ★ saved
   8/200  │   0.6396 │   0.7498 │ 0.6354 │ 0.5197 │ 0.5470 │ 0.4792 │  1.00e-04 │ ★ saved
   9/200  │   0.6160 │   0.7054 │ 0.6496 │ 0.5549 │ 0.5501 │ 0.4982 │  1.00e-04 │ ★ saved
  10/200  │   0.5942 │   0.6720 │ 0.6623 │ 0.5804 │ 0.5572 │ 0.5100 │  1.00e-04 │ ★ saved
  11/200  │   0.5770 │   0.6409 │ 0.6711 │ 0.6046 │ 0.5609 │ 0.5194 │  1.00e-04 │ ★ saved
  12/200  